# Euro500 Equity Portfolio

This notebook constructs a quarterly rebalanced equity portfolio consisting of the 500 largest non-financial firms headquartered in euro-area countries. The sample starts in 1999Q1 and is used as the basis for subsequent return and event-study analyses.

## 0. Setup

In [ ]:
# --- Imports & configuration ---
from pathlib import Path
import pandas as pd
import numpy as np
import lseg.data as ld
import time
import warnings

# --- Output paths (anpassen) ---
BASE_DIR = Path("/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data")
(DATA_DIR := BASE_DIR / "intermediate").mkdir(parents=True, exist_ok=True)

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

warnings.filterwarnings(
    "ignore",
    message=r".*Downcasting behavior in `replace` is deprecated.*",
    category=FutureWarning,
    module=r"lseg\.data\._tools\._dataframe"
)

## 1. Definition of the Investment Universe

### 1.1 Euro-Area Headquartered Firms

The initial universe consists of all publicly listed, active, primary equity instruments whose headquarters are located in a euro-area country.

In [69]:
EURO_HQ_CODES = [
    "AT","BE","CY","EE","FI","FR","DE","GR",
    "IE","IT","LV","LT","LU","MT","NL","PT",
    "SK","SI","ES", "HR"
]

hq_codes_str = ",".join(f'"{c}"' for c in EURO_HQ_CODES)

SCREEN_EURO_ALL = (
    "SCREEN("
    "U(IN(Equity(active,public,primary))),"
    f"IN(TR.HQCountryCode,{hq_codes_str}),"
    "CURN=EUR"
    ")"
)

### 1.2 Period

Every Quater from 1999 to 2025

In [42]:
# Quarter ends: 1999Q1 bis 2025Q4 (oder bis heute)
START = "1998-12-31"
END   = "2025-12-31"   # ggf. "2026-12-31" oder pd.Timestamp.today()

q_ends = pd.date_range(START, END, freq="Q")
q_ends[:5], q_ends[-5:]

/var/folders/r2/vtz74sz14fx185wt9m3c781w0000gn/T/ipykernel_63183/4226632070.py:5:FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.


(DatetimeIndex(['1998-12-31', '1999-03-31', '1999-06-30', '1999-09-30', '1999-12-31'], dtype='datetime64[ns]', freq='QE-DEC'),
 DatetimeIndex(['2024-12-31', '2025-03-31', '2025-06-30', '2025-09-30', '2025-12-31'], dtype='datetime64[ns]', freq='QE-DEC'))

In [4]:
EURO_COUNTRIES = [
    "Austria","Belgium","Croatia","Cyprus","Estonia","Finland","France","Germany",
    "Greece","Ireland","Italy","Latvia","Lithuania","Luxembourg","Malta","Netherlands",
    "Portugal","Slovakia","Slovenia","Spain", "Croatia"
]

def _rename_first_match(df: pd.DataFrame, candidates, new_name):
    for c in candidates:
        if c in df.columns:
            return df.rename(columns={c: new_name})
    return df

def _require_cols(df: pd.DataFrame, cols, context=""):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{context}] Missing columns: {missing}. Got: {list(df.columns)}")

In [ ]:
def make_country_filter(countries):
    quoted = ",".join([f"'{c}'" for c in countries])
    return quoted

## 2. Workspace Screener

In [ ]:
CACHE_DIR = DATA_DIR / "euro_snap_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _cache_path(date_iso: str) -> Path:
    return CACHE_DIR / f"euro_snapshot_{date_iso}.parquet"

def pull_euro_equities_snapshot_safe(date_iso: str, max_retries: int = 3, sleep_s: float = 2.0) -> pd.DataFrame:
    """
    Safe wrapper: retry + minimal field fallback.
    Returns standardized columns: RIC, name, hq_country, hq_code, trbc_sector_code, trbc_sector, mcap_eur
    """
    # --- cache first ---
    p = _cache_path(date_iso)
    if p.exists():
        return pd.read_parquet(p)

    universe = [SCREEN_EURO_ALL]

    # Strategy:
    # 1) try full fields incl CompanyMarketCap + MarketCap
    # 2) if backend 400, try only CompanyMarketCap
    # 3) if still, try only MarketCap
    field_sets = [
        ["TR.RIC","TR.CommonName","TR.HeadquartersCountry","TR.HQCountryCode",
         "TR.TRBCEconSectorCode","TR.TRBCEconomicSector","TR.CompanyMarketCap","TR.MarketCap"],
        ["TR.RIC","TR.CommonName","TR.HeadquartersCountry","TR.HQCountryCode",
         "TR.TRBCEconSectorCode","TR.TRBCEconomicSector","TR.CompanyMarketCap"],
        ["TR.RIC","TR.CommonName","TR.HeadquartersCountry","TR.HQCountryCode",
         "TR.TRBCEconSectorCode","TR.TRBCEconomicSector","TR.MarketCap"],
    ]

    params = {"CURN":"EUR","RH":"In","CH":"Fd","SDate":date_iso,"EDate":date_iso}

    last_err = None

    for fields in field_sets:
        for attempt in range(1, max_retries + 1):
            try:
                ld.open_session()
                try:
                    df = ld.get_data(universe=universe, fields=fields, parameters=params).copy()
                finally:
                    ld.close_session()

                # --- map your backend-friendly names (as observed) ---
                cols = list(df.columns)
                lower = {c.lower(): c for c in cols}

                def get_col(*names):
                    for n in names:
                        if n in df.columns:
                            return n
                        nl = n.lower()
                        if nl in lower:
                            return lower[nl]
                    return None

                col_ric = get_col("RIC","TR.RIC","Instrument")
                col_name = get_col("Company Common Name","TR.CommonName")
                col_hq_country = get_col("Country of Headquarters","TR.HeadquartersCountry")
                col_hq_code = get_col("Country ISO Code of Headquarters","TR.HQCountryCode")
                col_trbc_code = get_col("TRBC Economic Sector Code","TR.TRBCEconSectorCode")
                col_trbc_name = get_col("TRBC Economic Sector Name","TR.TRBCEconomicSector")
                col_mcap = get_col("Company Market Cap","TR.CompanyMarketCap","TR.MarketCap","Market Cap")

                if col_ric is None:
                    raise KeyError(f"No RIC column. Got: {cols}")
                if col_mcap is None:
                    raise KeyError(f"No MarketCap column. Got: {cols}")

                out = pd.DataFrame({
                    "RIC": df[col_ric],
                    "name": df[col_name] if col_name else pd.NA,
                    "hq_country": df[col_hq_country] if col_hq_country else pd.NA,
                    "hq_code": df[col_hq_code] if col_hq_code else pd.NA,
                    "trbc_sector_code": df[col_trbc_code] if col_trbc_code else pd.NA,
                    "trbc_sector": df[col_trbc_name] if col_trbc_name else pd.NA,
                    "mcap_eur": df[col_mcap],
                })

                out["mcap_eur"] = pd.to_numeric(out["mcap_eur"], errors="coerce")
                out = out.dropna(subset=["RIC"]).copy()

                # cache
                out.to_parquet(p, index=False)
                return out

            except Exception as e:
                last_err = e
                # simple backoff
                time.sleep(sleep_s * attempt)

    raise last_err

### 2.2 Loop über Screener um für jeder Quartal alle Equities zu ziehen

In [ ]:
def build_quarterly_euro_panel_safe(q_ends: pd.DatetimeIndex) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = []
    failures = []

    for dt in q_ends:
        date_iso = dt.strftime("%Y-%m-%d")
        print("Quarter:", dt.date(), "->", date_iso)

        try:
            snap = pull_euro_equities_snapshot_safe(date_iso)
            snap.insert(0, "date", pd.Timestamp(dt).normalize())
            out.append(snap)
        except Exception as e:
            failures.append({"date": date_iso, "error": str(e)[:500]})
            print(" ❌ failed:", str(e)[:200], "...")

    panel = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
    fail_df = pd.DataFrame(failures)
    return panel, fail_df

panel_all, fail_log = build_quarterly_euro_panel_safe(q_ends)

print("Done. Panel rows:", len(panel_all))
print("Failures:", len(fail_log))

Quarter: 1998-12-31 -> 1998-12-31
Quarter: 1999-03-31 -> 1999-03-31
Quarter: 1999-06-30 -> 1999-06-30
Quarter: 1999-09-30 -> 1999-09-30
Quarter: 1999-12-31 -> 1999-12-31
Quarter: 2000-03-31 -> 2000-03-31
Quarter: 2000-06-30 -> 2000-06-30
Quarter: 2000-09-30 -> 2000-09-30
Quarter: 2000-12-31 -> 2000-12-31
Quarter: 2001-03-31 -> 2001-03-31
Quarter: 2001-06-30 -> 2001-06-30
Quarter: 2001-09-30 -> 2001-09-30
Quarter: 2001-12-31 -> 2001-12-31
Quarter: 2002-03-31 -> 2002-03-31
Quarter: 2002-06-30 -> 2002-06-30
Quarter: 2002-09-30 -> 2002-09-30
Quarter: 2002-12-31 -> 2002-12-31
Quarter: 2003-03-31 -> 2003-03-31
Quarter: 2003-06-30 -> 2003-06-30
Quarter: 2003-09-30 -> 2003-09-30
Quarter: 2003-12-31 -> 2003-12-31
Quarter: 2004-03-31 -> 2004-03-31
Quarter: 2004-06-30 -> 2004-06-30
Quarter: 2004-09-30 -> 2004-09-30
Quarter: 2004-12-31 -> 2004-12-31
Quarter: 2005-03-31 -> 2005-03-31
Quarter: 2005-06-30 -> 2005-06-30
Quarter: 2005-09-30 -> 2005-09-30
Quarter: 2005-12-31 -> 2005-12-31
Quarter: 2006-

In [53]:
panel_all.to_parquet(DATA_DIR / "euro_equities_all_quarterly_snapshots.parquet", index=False)
fail_log.to_csv(DATA_DIR / "euro_equities_pull_failures.csv", index=False)

### 2.3 Top 500 in jedem Quartal behalten

In [59]:
# Financials ausschließen (string-basiert, robust)
panel_nonfin = panel_all[
    panel_all["trbc_sector"]
    .astype(str)
    .str.strip()
    .str.lower()
    != "financials"
].copy()

panel_nonfin = panel_nonfin.sort_values(
    ["date", "mcap_eur", "RIC"],
    ascending=[True, False, True]
)

panel_nonfin["rank_mcap"] = (
    panel_nonfin
    .groupby("date")
    .cumcount()
    + 1
)

euro500 = panel_nonfin[panel_nonfin["rank_mcap"] <= 500].copy()

In [ ]:
# 1) Genau 500 Firmen pro Quartal?
print(euro500.groupby("date")["RIC"].nunique().describe())

count    109.0
mean     500.0
std        0.0
min      500.0
25%      500.0
50%      500.0
75%      500.0
max      500.0
Name: RIC, dtype: float64
